# 05-8. pandas와 DataFrame — 풀이 검증

## Goal

오류 행을 보존하며 전체 실행과 청크 실행 결과를 대조한다.

> 학습자용 TODO를 먼저 완성한 뒤 참고한다.


## Setup

fixture와 실행 환경을 확인한다.


In [ ]:
from pathlib import Path
import sys


def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "requirements.txt").is_file():
            return candidate
    raise FileNotFoundError("requirements.txt가 있는 저장소 루트에서 JupyterLab을 실행하세요.")


ROOT = find_project_root()
FIXTURE_DIR = ROOT / "fixtures" / "05-text-processing"

assert sys.version_info >= (3, 10)
assert FIXTURE_DIR.is_dir()

print("Python:", sys.version.split()[0])
print("실습 데이터:", FIXTURE_DIR)


import numpy as np
import pandas as pd
fixture_path = FIXTURE_DIR / "items.csv"
ITEM_DTYPES = {
    "name": "string",
    "category": "string",
    "price": "string",
    "quantity": "string",
}
CSV_READ_OPTIONS = {
    "dtype": ITEM_DTYPES,
    "encoding": "utf-8",
    "encoding_errors": "strict",
    "keep_default_na": False,
    "na_values": [""],
}
frame = pd.read_csv(fixture_path, **CSV_READ_OPTIONS)


## Steps

참고 구현을 실행한다.


In [ ]:
def clean_items(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    working = frame.copy()
    working["price_number"] = pd.to_numeric(working["price"], errors="coerce")
    working["quantity_number"] = pd.to_numeric(working["quantity"], errors="coerce")

    price_values = working["price_number"].to_numpy(dtype=float, na_value=np.nan)
    quantity_values = working["quantity_number"].to_numpy(dtype=float, na_value=np.nan)
    missing_name = working["name"].astype("string").str.strip().fillna("").eq("").to_numpy()
    invalid_price = ~np.isfinite(price_values) | (price_values < 0)
    invalid_quantity = (
        ~np.isfinite(quantity_values)
        | (quantity_values < 0)
        | np.not_equal(quantity_values, np.floor(quantity_values))
    )
    reasons = np.select(
        [missing_name, invalid_price, invalid_quantity],
        ["missing_name", "invalid_price", "invalid_quantity"],
        default="",
    )
    invalid = pd.Series(reasons != "", index=working.index)
    error_rows = working.loc[invalid].copy()
    error_rows["error_reason"] = reasons[invalid.to_numpy()]
    valid_rows = working.loc[~invalid].copy()
    valid_rows["name"] = valid_rows["name"].astype("string").str.strip()
    valid_rows["category"] = (
        valid_rows["category"].astype("string").str.strip().replace("", pd.NA).fillna("UNKNOWN")
    )
    valid_rows["total"] = valid_rows["price_number"] * valid_rows["quantity_number"]
    summary = valid_rows.groupby("category", dropna=False).agg(
        item_count=("name", "count"),
        total_amount=("total", "sum"),
    ).reset_index()
    return valid_rows, error_rows, summary


valid_rows, error_rows, summary = clean_items(frame)
summary


## Checks

경계값과 fixture 결과를 대조한다.


In [ ]:
assert len(valid_rows) == 4 and len(error_rows) == 2
assert int(summary["item_count"].sum()) == 4

boundary = pd.DataFrame([
    {"name": "valid", "category": "input", "price": "1", "quantity": "1"},
    {"name": "infinite", "category": "input", "price": "inf", "quantity": "1"},
    {"name": "negative", "category": "input", "price": "-1", "quantity": "1"},
    {"name": "fractional", "category": "input", "price": "1", "quantity": "1.5"},
    {"name": "  ", "category": "input", "price": "1", "quantity": "1"},
])
boundary_valid, boundary_errors, _ = clean_items(boundary)
assert len(boundary_valid) == 1 and len(boundary_errors) == 4
assert set(boundary_errors["error_reason"]) == {
    "missing_name", "invalid_price", "invalid_quantity"
}

large_path = FIXTURE_DIR / "large_items.csv"
full_large = pd.read_csv(large_path, **CSV_READ_OPTIONS)
full_valid, full_errors, full_summary = clean_items(full_large)

chunk_valid = chunk_errors = 0
chunk_summaries = []
for chunk in pd.read_csv(large_path, chunksize=37, **CSV_READ_OPTIONS):
    valid, errors, chunk_summary = clean_items(chunk)
    chunk_valid += len(valid)
    chunk_errors += len(errors)
    chunk_summaries.append(chunk_summary)

combined_summary = (
    pd.concat(chunk_summaries, ignore_index=True)
    .groupby("category", as_index=False)[["item_count", "total_amount"]]
    .sum()
    .sort_values("category")
    .reset_index(drop=True)
)
expected_summary = full_summary.sort_values("category").reset_index(drop=True)

assert chunk_valid == 196 and chunk_errors == 4
assert len(full_valid) == chunk_valid and len(full_errors) == chunk_errors
pd.testing.assert_frame_equal(expected_summary, combined_summary, check_dtype=False)
print("검증 통과:", chunk_valid, "정상 /", chunk_errors, "오류")


## Next Steps

청크를 모두 저장하지 말고 최종 집계와 오류 건수만 증분 누적한다.
